In [1]:
import fastf1

fastf1.Cache.enable_cache("../data/cache")

In [2]:
session = fastf1.get_session(2024, "Monza", "Q")

session.load()

core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '63', '16', '55', '44', '1', '11', '23', '27', '14', '3', '20', '10', '31', '22', '18', '43', '77', '24']


In [5]:
lap = session.laps.pick_drivers("LEC").pick_fastest().get_telemetry()
tel = lap

In [6]:
import pandas as pd
export = pd.DataFrame({
    "time":     tel["Time"].dt.total_seconds(),
    "x":        tel["X"] / 10,
    "y":        tel["Y"] / 10,
    "z":        tel["Z"] / 10,
    "speed":    tel["Speed"],
    "throttle": tel["Throttle"],
    "brake":    tel["Brake"].astype(int),
    "gear":     tel["nGear"],
})
# Make time start at 0
export["time"] = export["time"] - export["time"].iloc[0]

export.head()

,time,x,y,z,speed,throttle,brake,gear
2,0.000,-137.291789,-68.119849,187.192991,322.0,99.617857,0,8
3,0.073,-136.900000,-61.900000,187.200000,322.0,99.357143,0,8
4,0.093,-136.800000,-60.200000,187.200000,322.0,99.285714,0,8
5,0.173,-136.365697,-53.855903,187.200364,322.0,99.000000,0,8
6,0.412,-134.427733,-33.574831,187.261810,325.0,100.000000,0,8


In [7]:
export.to_csv("../data/samples/f1_monza_lec_q.csv", index=False)

In [8]:
%load_ext autoreload
%autoreload 2
from telemetry.f1 import load_session
session = load_session(2024, "Monza", "R")
session.laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:57:18.931000,LEC,16,0 days 00:01:28.179000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:29.989000,...,True,Ferrari,0 days 00:55:50.494000,2024-09-01 13:03:34.413,1,2.0,False,,False,False
1,0 days 00:58:44.327000,LEC,16,0 days 00:01:25.396000,2.0,1.0,NaT,NaT,0 days 00:00:27.707000,0 days 00:00:29.265000,...,True,Ferrari,0 days 00:57:18.931000,2024-09-01 13:05:02.850,1,2.0,False,,False,True
2,0 days 01:00:09.506000,LEC,16,0 days 00:01:25.179000,3.0,1.0,NaT,NaT,0 days 00:00:27.679000,0 days 00:00:29.001000,...,True,Ferrari,0 days 00:58:44.327000,2024-09-01 13:06:28.246,1,2.0,False,,False,True
3,0 days 01:01:34.316000,LEC,16,0 days 00:01:24.810000,4.0,1.0,NaT,NaT,0 days 00:00:27.653000,0 days 00:00:28.883000,...,True,Ferrari,0 days 01:00:09.506000,2024-09-01 13:07:53.425,1,2.0,False,,False,True
4,0 days 01:02:58.919000,LEC,16,0 days 00:01:24.603000,5.0,1.0,NaT,NaT,0 days 00:00:27.630000,0 days 00:00:28.790000,...,True,Ferrari,0 days 01:01:34.316000,2024-09-01 13:09:18.235,1,2.0,False,,False,True


In [9]:
from telemetry.f1 import build_race_replay

replay = build_race_replay(session)

print("Rows:", len(replay))
print("Drivers:", replay["vehicle_id"].nunique())
print("Time range:", replay["time"].min(), "to", replay["time"].max())

Rows: 360500
Drivers: 20
Time range: -59.96399999999994 to 4619.957


In [10]:
from telemetry.f1 import save_replay, load_replay, REPLAYS_DIR

path = save_replay(replay, REPLAYS_DIR / "monza_2024_race.parquet")
print("Saved to:", path)
print("File size:", round(path.stat().st_size / 1_000_000, 1), "MB")

loaded = load_replay(path)
print("Same shape:", loaded.shape == replay.shape)
print("Same types:", (loaded.dtypes == replay.dtypes).all())

Saved to: /Users/varun/Desktop/Projects/telemetry_platform/data/replays/monza_2024_race.parquet
File size: 3.1 MB
Same shape: True
Same types: True
